This notebook just contains some exploration of the dataset.

In [1]:
import sys

# Necessary to import from src dir
sys.path.append('..')

In [2]:
import matplotlib.pyplot as mlp
import numpy as np
import os
import pandas as pd

from sklearn.preprocessing import StandardScaler

from src.preprocessing import (
    prepare_standardized_datasets,
    standardize_data
)

In [3]:
# Note: notebooks/data is ignored by git, so any files in there will not end up in version control
DATA_DIR = r"./data/"
OUTPUT_DIR = r"./output/"
TEMP_OUTPUT_DIR = r"./temp/"

TRAIN_DATA = os.path.join(DATA_DIR, "train.csv")
TEST_DATA = os.path.join(DATA_DIR, "test.csv")

# Preprocessing

df_train = pd.read_csv(TRAIN_DATA)
df_test = pd.read_csv(TEST_DATA)

label_var = "class4"

# Include only the real-valued mean values, but don't exclude anything further than that yet before exploring the data a bit first
cols_to_include = [feat for feat in df_train.columns.to_list() if (feat.endswith(".mean")) or (feat == label_var)]

numeric_vars = cols_to_include[:]
numeric_vars.remove(label_var)

In [4]:
# Check the Pearson correlation coefficients for the data
# Maybe we can eliminate some redundant variables already at this point

corr = df_train[cols_to_include].corr(numeric_only=True)
corr

,CO2168.mean,CO2336.mean,CO242.mean,CO2504.mean,Glob.mean,H2O168.mean,H2O336.mean,H2O42.mean,H2O504.mean,H2O672.mean,...,SO2168.mean,SWS.mean,T168.mean,T42.mean,T504.mean,T672.mean,T84.mean,UV_A.mean,UV_B.mean,CS.mean
CO2168.mean,1.000000,0.999675,0.993428,0.998567,-0.422862,-0.428546,-0.424609,-0.434061,-0.423042,-0.421750,...,0.313204,-0.144759,-0.565241,-0.563994,-0.568525,-0.567936,-0.564454,-0.434419,-0.434426,-0.106144
CO2336.mean,0.999675,1.000000,0.992175,0.999533,-0.418366,-0.434089,-0.430156,-0.439643,-0.428607,-0.427356,...,0.310494,-0.140993,-0.565057,-0.563768,-0.568681,-0.568213,-0.564217,-0.429937,-0.430238,-0.110800
CO242.mean,0.993428,0.992175,1.000000,0.989651,-0.410648,-0.371312,-0.367937,-0.375666,-0.366559,-0.365313,...,0.316664,-0.157891,-0.523636,-0.523111,-0.526157,-0.525160,-0.523352,-0.416266,-0.406660,-0.082900
CO2504.mean,0.998567,0.999533,0.989651,1.000000,-0.415815,-0.441403,-0.437480,-0.447008,-0.435965,-0.434780,...,0.305035,-0.138026,-0.565800,-0.564382,-0.569942,-0.569724,-0.564860,-0.427277,-0.428167,-0.118085
Glob.mean,-0.422862,-0.418366,-0.410648,-0.415815,1.000000,0.162005,0.156354,0.172892,0.152518,0.151024,...,-0.115704,0.369409,0.594203,0.592415,0.589929,0.589997,0.596149,0.985450,0.935493,0.122259
H2O168.mean,-0.428546,-0.434089,-0.371312,-0.441403,0.162005,1.000000,0.999883,0.999604,0.999704,0.999495,...,-0.242150,-0.239006,0.815735,0.816751,0.820090,0.821259,0.814638,0.252725,0.371318,0.476726
H2O336.mean,-0.424609,-0.430156,-0.367937,-0.437480,0.156354,0.999883,1.000000,0.999138,0.999937,0.999789,...,-0.241652,-0.241590,0.812800,0.813907,0.817181,0.818372,0.811749,0.246731,0.365475,0.478715
H2O42.mean,-0.434061,-0.439643,-0.375666,-0.447008,0.172892,0.999604,0.999138,1.000000,0.998759,0.998440,...,-0.243284,-0.235336,0.820766,0.821548,0.825134,0.826284,0.819552,0.263820,0.382417,0.472611
H2O504.mean,-0.423042,-0.428607,-0.366559,-0.435965,0.152518,0.999704,0.999937,0.998759,1.000000,0.999901,...,-0.241047,-0.242402,0.810517,0.811649,0.814972,0.816216,0.809472,0.242609,0.361413,0.479612
H2O672.mean,-0.421750,-0.427356,-0.365313,-0.434780,0.151024,0.999495,0.999789,0.998440,0.999901,1.000000,...,-0.239400,-0.243683,0.808509,0.809640,0.813044,0.814364,0.807462,0.240687,0.359400,0.480559


In [5]:
def get_n_largest_pearson_coeffs_for_variable(
    corr,
    variable,
    n=10
):
    return np.abs(corr[variable]).nlargest(n)

In [6]:
# We can get the indices of the variables with which the correlation is largest
get_n_largest_pearson_coeffs_for_variable(corr, "CO2168.mean").index

Index(['CO2168.mean', 'CO2336.mean', 'CO2504.mean', 'CO242.mean', 'T504.mean',
       'T672.mean', 'T168.mean', 'T84.mean', 'T42.mean', 'UV_B.mean'],
      dtype='object')

In [7]:
# Or just view the values
get_n_largest_pearson_coeffs_for_variable(corr, "CO2168.mean")

CO2168.mean    1.000000
CO2336.mean    0.999675
CO2504.mean    0.998567
CO242.mean     0.993428
T504.mean      0.568525
T672.mean      0.567936
T168.mean      0.565241
T84.mean       0.564454
T42.mean       0.563994
UV_B.mean      0.434426
Name: CO2168.mean, dtype: float64

Looking at the covariances, we notice very quickly that for $CO_2$ values, it's probably enough to include just one.

In [8]:
# Figure out the columns with the highest overall correlation coefficients
# by calculating the sum of the absolute value of the correlation coefficient table
# and sorting to find those that might be most correlated with others.

n_pearsons_sum_per_col = []

for col in corr.columns:
    n_pearsons_sum_per_col.append(get_n_largest_pearson_coeffs_for_variable(corr, col, len(corr.columns)).sum())

exploratory_table = corr.copy()
exploratory_table["sum_pearson"] = n_pearsons_sum_per_col

exploratory_table["sum_pearson"].nlargest(50)

T84.mean          26.307055
T168.mean         26.270590
T42.mean          26.267059
T504.mean         26.200806
T672.mean         26.170127
UV_B.mean         26.060934
UV_A.mean         26.026486
NET.mean          25.596835
PAR.mean          25.451764
Glob.mean         25.005143
RHIRGA336.mean    24.933839
RHIRGA504.mean    24.930007
RHIRGA672.mean    24.814426
RHIRGA168.mean    24.595503
RHIRGA84.mean     24.303401
RHIRGA42.mean     24.212245
NOx42.mean        22.681271
NOx84.mean        22.673173
NOx168.mean       22.673092
NOx336.mean       22.570084
NOx504.mean       22.499140
NOx672.mean       22.411238
O3672.mean        21.114269
O3504.mean        20.941694
O3168.mean        20.288422
CO2168.mean       20.049462
O384.mean         20.004236
CO2336.mean       19.996899
CO2504.mean       19.946889
O342.mean         19.699757
CO242.mean        19.340127
NO336.mean        19.226499
NO504.mean        19.158918
NO672.mean        19.124453
NO168.mean        19.054896
NO84.mean         18

In [9]:
# If nothing else, we at least may want to keep the variables towards the low end for a longer while

In [10]:
# Prepare standardized datasets

df_train, df_validation, df_test = prepare_standardized_datasets(
    df_train=df_train,
    df_test=df_test,
    data_vars=numeric_vars,
    label_var=label_var,
    label_values=("nonevent", "event"),
    include_validation_split=True,
    validation_split_size=0.2
)


In [14]:
df_train, _, df_test = prepare_standardized_datasets(
    df_train=df_train,
    df_test=df_test,
    data_vars=numeric_vars,
    label_var=label_var,
    label_values=("nonevent", "event"),
    include_validation_split=False,
    validation_split_size=0.2
)

In [15]:
df_train.head()

,class4,CO2168.mean,CO2336.mean,CO242.mean,CO2504.mean,Glob.mean,H2O168.mean,H2O336.mean,H2O42.mean,H2O504.mean,...,SO2168.mean,SWS.mean,T168.mean,T42.mean,T504.mean,T672.mean,T84.mean,UV_A.mean,UV_B.mean,CS.mean
0,event,-0.843284,-0.806178,-0.749904,-0.779346,0.851016,-0.381931,-0.390005,-0.326875,-0.395056,...,-0.535031,0.414331,0.258275,0.195343,0.243301,0.242919,0.220021,0.463568,0.404275,-0.690468
1,event,-0.909258,-0.901013,-0.957330,-0.885940,-0.678721,0.995845,1.003319,0.971381,1.010501,...,-0.588183,0.344518,0.530734,0.542361,0.525037,0.525596,0.532190,-0.447675,-0.221725,0.121596
2,event,-0.788545,-0.858080,-0.718858,-0.954125,-0.996964,0.675420,0.679352,0.669455,0.677365,...,-0.600890,-0.550207,0.217902,0.244002,0.223605,0.233052,0.216188,-0.891703,-0.884774,-0.669991
3,event,-1.450075,-1.437434,-1.546171,-1.424263,0.684215,0.073630,0.071292,0.095712,0.071972,...,0.651056,0.433459,0.521701,0.486989,0.544710,0.539018,0.518784,0.220481,0.423714,1.283602
4,event,0.002070,0.019693,0.086694,0.034396,0.847362,1.982168,1.997591,1.953313,2.005163,...,0.289840,0.264095,1.550111,1.558378,1.539554,1.540126,1.556622,0.903245,1.467806,1.667170


In [16]:
df_test.head()

,CO2168.mean,CO2336.mean,CO242.mean,CO2504.mean,Glob.mean,H2O168.mean,H2O336.mean,H2O42.mean,H2O504.mean,H2O672.mean,...,SO2168.mean,SWS.mean,T168.mean,T42.mean,T504.mean,T672.mean,T84.mean,UV_A.mean,UV_B.mean,CS.mean
0,-1.118472,-1.132191,-1.158624,-1.130211,1.200503,1.351899,1.353238,1.339886,1.357780,1.360061,...,-0.472452,0.381289,1.326148,1.339288,1.320454,1.323639,1.330858,1.379542,1.634996,0.376366
1,-0.973844,-0.952001,-1.057652,-0.937878,1.352984,-0.461841,-0.466771,-0.452994,-0.466866,-0.466333,...,-0.399677,0.410935,0.637692,0.630600,0.640245,0.650919,0.640302,1.143024,0.992869,1.253726
2,0.697000,0.697619,0.687074,0.696291,-1.141205,-0.851818,-0.836964,-0.864219,-0.845616,-0.830904,...,2.810646,0.102848,-1.189380,-1.171034,-1.195394,-1.201929,-1.177207,-1.057774,-1.128053,1.919376
3,1.345761,1.347395,1.317112,1.346704,0.183713,-1.171722,-1.177013,-1.161082,-1.179224,-1.124627,...,-0.316036,0.298544,-1.568541,-1.575249,-1.587386,-1.600795,-1.566562,0.051622,-0.525677,-0.939134
4,-1.020092,-1.004697,-1.009901,-1.004807,0.813245,0.426985,0.418221,0.438750,0.413230,0.411642,...,-0.445689,0.610007,0.728378,0.719375,0.730776,0.732718,0.718298,1.115702,0.966431,-0.472808
